## Convert Bounding Box to Segmentation Mask

In [6]:
# YOLOv5 Object Counter
# By: GitHub Copilot for mmaleki92 (2025-02-27)

import os
import sys
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image, clear_output
from google.colab import files
from pathlib import Path
import time
import urllib.request

# Install YOLOv5 if not already installed
def install_yolov5():
    if not os.path.exists('yolov5'):
        print("Installing YOLOv5...")
        # Clone YOLOv5 repository
        !git clone https://github.com/ultralytics/yolov5
        # Install YOLOv5 requirements
        !pip install -q -r yolov5/requirements.txt
        # Add yolov5 directory to path
        sys.path.append('./yolov5')
        print("YOLOv5 installed successfully!")
    else:
        print("YOLOv5 is already installed.")
        sys.path.append('./yolov5')

# Function to load YOLOv5 model
def load_model(model_size='s'):
    # Valid model sizes: n, s, m, l, x
    valid_sizes = ['n', 's', 'm', 'l', 'x']
    if model_size not in valid_sizes:
        print(f"Invalid model size. Using 's' instead. Valid sizes are: {', '.join(valid_sizes)}")
        model_size = 's'

    model = torch.hub.load('ultralytics/yolov5', f'yolov5{model_size}', pretrained=True)
    return model

# Function to count objects in an image
def count_objects(results):
    # Get detected objects
    df = results.pandas().xyxy[0]

    # Count objects by class
    object_counts = {}
    for class_name in df['name'].unique():
        count = len(df[df['name'] == class_name])
        object_counts[class_name] = count

    return object_counts, df

# Function to draw bounding boxes and counts on image
def draw_boxes_and_counts(img, df, counts):
    # Make a copy of the image to avoid modifying the original
    img_with_boxes = img.copy()

    # Draw each bounding box
    for idx, row in df.iterrows():
        xmin, ymin, xmax, ymax = int(row['xmin']), int(row['ymin']), int(row['xmax']), int(row['ymax'])
        class_name = row['name']
        confidence = row['confidence']

        # Generate a consistent color for this class
        color = (int(hash(class_name) % 255),
                 int(hash(class_name + '1') % 255),
                 int(hash(class_name + '2') % 255))

        # Draw rectangle and label
        cv2.rectangle(img_with_boxes, (xmin, ymin), (xmax, ymax), color, 2)
        label = f"{class_name}: {confidence:.2f}"
        cv2.putText(img_with_boxes, label, (xmin, ymin - 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Add count information at the top
    y_pos = 30
    for class_name, count in counts.items():
        text = f"{class_name}: {count}"
        cv2.putText(img_with_boxes, text, (10, y_pos),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        y_pos += 30

    return img_with_boxes

# Process image file
def process_image(model, image_path, conf_threshold=0.25):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error loading image from {image_path}")
        return None, None

    # Convert BGR to RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Set confidence threshold
    model.conf = conf_threshold

    # Perform inference
    results = model(img_rgb)

    # Count objects
    counts, detections = count_objects(results)

    # Draw boxes and counts
    img_with_boxes = draw_boxes_and_counts(img_rgb, detections, counts)

    return img_with_boxes, counts

# Process video file
def process_video(model, video_path, output_path=None, conf_threshold=0.25):
    # Open video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file {video_path}")
        return None

    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Create output video writer if output path is specified
    writer = None
    if output_path:
        writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    # Set confidence threshold
    model.conf = conf_threshold

    frame_count = 0
    start_time = time.time()
    all_counts = {}

    # Process each frame
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Perform inference
        results = model(frame_rgb)

        # Count objects
        counts, detections = count_objects(results)

        # Update total counts
        for class_name, count in counts.items():
            if class_name in all_counts:
                all_counts[class_name] = max(all_counts[class_name], count)
            else:
                all_counts[class_name] = count

        # Draw boxes and counts
        frame_with_boxes = draw_boxes_and_counts(frame_rgb, detections, counts)

        # Convert RGB back to BGR for video writing
        frame_with_boxes_bgr = cv2.cvtColor(frame_with_boxes, cv2.COLOR_RGB2BGR)

        if writer:
            writer.write(frame_with_boxes_bgr)

        # Display progress
        frame_count += 1
        if frame_count % 30 == 0:  # Update every 30 frames
            elapsed_time = time.time() - start_time
            fps_processed = frame_count / elapsed_time
            print(f"Processed {frame_count} frames. FPS: {fps_processed:.2f}", end="\r")

    # Release resources
    cap.release()
    if writer:
        writer.release()

    print(f"\nVideo processing complete. Processed {frame_count} frames.")
    return all_counts

# Function to download an example image or video
def download_example_file(url, filename):
    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(url, filename)
    print(f"Downloaded to {filename}")
    return filename

# Main function
def main():
    # Install YOLOv5
    install_yolov5()

    # Load model
    print("Loading YOLOv5 model...")
    model = load_model('s')  # Load the small model
    print("Model loaded successfully!")

    # Ask user for input type
    print("\nWhat would you like to process?")
    print("1. Example image (street scene)")
    print("2. Example video (traffic)")
    print("3. Upload your own image")
    print("4. Upload your own video")

    choice = input("Enter your choice (1-4): ")

    if choice == '1':
        # Download example image
        image_url = "https://ultralytics.com/images/zidane.jpg"
        image_path = download_example_file(image_url, "example_image.jpg")

        # Process image
        result_img, counts = process_image(model, image_path)

        # Display results
        plt.figure(figsize=(12, 8))
        plt.imshow(result_img)
        plt.title("Object Detection Results")
        plt.axis('off')
        plt.show()

        print("\nObject counts:")
        for class_name, count in counts.items():
            print(f"{class_name}: {count}")

    elif choice == '2':
        # Download example video
        video_url = "https://ultralytics.com/assets/highway.mp4"
        video_path = download_example_file(video_url, "example_video.mp4")
        output_path = "result_video.mp4"

        # Process video
        counts = process_video(model, video_path, output_path)

        print("\nMaximum object counts throughout the video:")
        for class_name, count in counts.items():
            print(f"{class_name}: {count}")

        print(f"\nProcessed video saved as {output_path}")

    elif choice == '3':
        # Upload image
        print("Please upload an image...")
        uploaded = files.upload()

        if uploaded:
            image_path = next(iter(uploaded.keys()))

            # Process image
            result_img, counts = process_image(model, image_path)

            # Display results
            plt.figure(figsize=(12, 8))
            plt.imshow(result_img)
            plt.title("Object Detection Results")
            plt.axis('off')
            plt.show()

            print("\nObject counts:")
            for class_name, count in counts.items():
                print(f"{class_name}: {count}")

            # Save and provide the result
            result_path = "detection_result.jpg"
            cv2.imwrite(result_path, cv2.cvtColor(result_img, cv2.COLOR_RGB2BGR))
            print(f"\nResult saved as {result_path}")
            files.download(result_path)

    elif choice == '4':
        # Upload video
        print("Please upload a video...")
        uploaded = files.upload()

        if uploaded:
            video_path = next(iter(uploaded.keys()))
            output_path = "result_video.mp4"

            # Process video
            counts = process_video(model, video_path, output_path)

            print("\nMaximum object counts throughout the video:")
            for class_name, count in counts.items():
                print(f"{class_name}: {count}")

            print(f"\nProcessed video saved as {output_path}")
            files.download(output_path)

    else:
        print("Invalid choice. Please run again and select a valid option.")

if __name__ == "__main__":
    main()

Installing YOLOv5...
Cloning into 'yolov5'...
remote: Enumerating objects: 17270, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 17270 (delta 0), reused 0 (delta 0), pack-reused 17269 (from 2)
Receiving objects: 100% (17270/17270), 16.11 MiB | 18.55 MiB/s, done.
Resolving deltas: 100% (11861/11861), done.
YOLOv5 installed successfully!
Loading YOLOv5 model...


You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /root/.cache/torch/hub/master.zip
YOLOv5 🚀 2025-2-27 Python-3.11.11 torch-2.5.1+cu124 CPU

100%|██████████| 14.1M/14.1M [00:00<00:00, 92.8MB/s]

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


Model loaded successfully!

What would you like to process?
1. Example image (street scene)
2. Example video (traffic)
3. Upload your own image
4. Upload your own video
Enter your choice (1-4): 4
Please upload a video...


Saving background video _ people _ walking _(1).mp4 to background video _ people _ walking _(1).mp4


`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(arg

...

`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(arg

`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(arg


Video processing complete. Processed 341 frames.

Maximum object counts throughout the video:
person: 42
handbag: 5
backpack: 2
skis: 1
dog: 1
skateboard: 2
suitcase: 1
snowboard: 1
chair: 1
umbrella: 1

Processed video saved as result_video.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>